In [5]:
from datetime import datetime

batch_id = datetime.utcnow().strftime("%Y%m%d_%H%M%S")
print("batch_id is: ", batch_id)

StatementMeta(, b93170a4-8152-46b9-b596-7554c86c1eee, 7, Finished, Available, Finished, False)

batch_id is:  20260313_115915


In [6]:
from pyspark.sql import functions as F

raw_tip_path = "Files/yelp/raw/yelp_tip.json"
df_tip_raw = spark.read.json(raw_tip_path)
df_tip_raw.printSchema()
print("raw tip count:", df_tip_raw.count())


StatementMeta(, b93170a4-8152-46b9-b596-7554c86c1eee, 8, Finished, Available, Finished, False)

root
 |-- business_id: string (nullable = true)
 |-- compliment_count: long (nullable = true)
 |-- date: string (nullable = true)
 |-- text: string (nullable = true)
 |-- user_id: string (nullable = true)

raw tip count: 908915


In [7]:
df_tip_bronze = (
    df_tip_raw
    .withColumn("_ingest_ts", F.current_date())
    .withColumn("_source_file",F.input_file_name())
    .withColumn("_batch_id",F.lit(batch_id))
)

StatementMeta(, b93170a4-8152-46b9-b596-7554c86c1eee, 9, Finished, Available, Finished, False)

In [8]:
(
    df_tip_bronze.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("tip_bronze")
)

StatementMeta(, b93170a4-8152-46b9-b596-7554c86c1eee, 10, Finished, Available, Finished, False)

In [9]:
df_tip_bronze.select("business_id","compliment_count","date","text","user_id")


StatementMeta(, b93170a4-8152-46b9-b596-7554c86c1eee, 11, Finished, Available, Finished, False)

DataFrame[business_id: string, compliment_count: bigint, date: string, text: string, user_id: string]

In [10]:
spark.table("tip_bronze").printSchema()

StatementMeta(, b93170a4-8152-46b9-b596-7554c86c1eee, 12, Finished, Available, Finished, False)

root
 |-- business_id: string (nullable = true)
 |-- compliment_count: long (nullable = true)
 |-- date: string (nullable = true)
 |-- text: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- _ingest_ts: date (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _batch_id: string (nullable = true)



In [11]:
#f返回结果给总控 01_0_bronze_run_all
mssparkutils.notebook.exit("SUCCESS")

StatementMeta(, b93170a4-8152-46b9-b596-7554c86c1eee, 13, Finished, Available, Finished, False)

ExitValue: SUCCESS